# Exploratory Data Analysis

This notebook reproduces the main EDA figures used in the README.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data')
IMAGE_DIR = Path('../image')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train_dataset.csv').replace(-9999, np.nan)
test = pd.read_csv(DATA_DIR / 'test_dataset.csv').replace(-9999, np.nan)

print('Train shape:', train.shape)
print('Test shape :', test.shape)
train.head()

In [ ]:
def get_hourly_cols(df, base):
    return [f'{base}_{h}' for h in range(24) if f'{base}_{h}' in df.columns]

def base_variables(df):
    bases = set()
    for c in df.columns:
        parts = c.split('_')
        if len(parts) >= 2:
            try:
                h = int(parts[-1])
                if 0 <= h <= 23:
                    bases.add('_'.join(parts[:-1]))
            except ValueError:
                pass
    return sorted(bases)

def add_date_features(df):
    out = df.copy()
    d = pd.to_datetime('2020-' + out['date'].astype(str), errors='coerce')
    out['month'] = d.dt.month
    out['dayofyear'] = d.dt.dayofyear
    return out

def add_wet_dry(df):
    out = df.copy()
    pcols = get_hourly_cols(out, 'precipitation')
    scols = get_hourly_cols(out, 'snow_depth')
    wet = out[pcols].fillna(0).gt(0).any(axis=1) | out[scols].fillna(0).gt(0).any(axis=1)
    out['weather_group'] = np.where(wet, 'Wet', 'Dry')
    return out

train_eda = add_wet_dry(add_date_features(train))
test_eda = add_wet_dry(add_date_features(test))
bases = base_variables(train_eda)
bases

In [ ]:
# Target statistics
train_eda['target'].agg(['count', 'mean', 'std', 'min', 'max']).to_frame('target')

In [ ]:
# Climatology seasonality
clim = train_eda.groupby('dayofyear', as_index=False)['climatology_temp'].mean()
plt.figure(figsize=(12, 5))
sns.lineplot(data=clim, x='dayofyear', y='climatology_temp', linewidth=2)
plt.title('Seasonal Pattern of Climatology Temperature')
plt.xlabel('Day of Year')
plt.ylabel('Climatology Temperature (°C)')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_climatology_seasonality.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Target residual by month
plt.figure(figsize=(12, 5))
sns.boxplot(data=train_eda, x='month', y='target', showfliers=False)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.title('Monthly Distribution of Target Residual')
plt.xlabel('Month')
plt.ylabel('Target Residual')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_target_residual_by_month.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Missing ratio by feature group
rows = []
for base in bases:
    cols = get_hourly_cols(train_eda, base)
    if cols:
        rows.append({'feature_group': base, 'missing_ratio': train_eda[cols].isna().mean().mean() * 100})
miss = pd.DataFrame(rows).sort_values('missing_ratio', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=miss, y='feature_group', x='missing_ratio')
plt.title('Average Missing Ratio by Weather Variable')
plt.xlabel('Missing Ratio (%)')
plt.ylabel('Feature Group')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_missing_ratio.png', dpi=300, bbox_inches='tight')
plt.show()
miss.head(20)

In [ ]:
# Correlation heatmap using daily mean features
corr_df = train_eda.copy()
features = []
for base in bases:
    cols = get_hourly_cols(corr_df, base)
    if cols:
        new_col = f'{base}_daily_mean'
        corr_df[new_col] = corr_df[cols].mean(axis=1)
        features.append(new_col)
features += ['climatology_temp', 'target']
features = [c for c in features if c in corr_df.columns and corr_df[c].notna().mean() > 0.3]
corr = corr_df[features].corr()

labels = {c: c.replace('_daily_mean', '').replace('_', '\n') for c in features}
plt.figure(figsize=(13, 11))
sns.heatmap(corr.rename(index=labels, columns=labels), cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Daily Mean Weather Variables')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_variable_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Target correlation bar plot
target_corr = corr['target'].drop('target').dropna().sort_values(key=lambda x: x.abs(), ascending=False)
plot_df = target_corr.head(20).reset_index()
plot_df.columns = ['feature', 'correlation']
plot_df['feature'] = plot_df['feature'].str.replace('_daily_mean', '', regex=False).str.replace('_', ' ', regex=False)
plot_df = plot_df.sort_values('correlation')

plt.figure(figsize=(9, 7))
sns.barplot(data=plot_df, x='correlation', y='feature')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Top Feature Correlations with Target')
plt.xlabel('Pearson Correlation with Target')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_target_correlation_bar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Wet / dry target distribution
plt.figure(figsize=(10, 5))
sns.kdeplot(data=train_eda, x='target', hue='weather_group', fill=True, common_norm=False, alpha=0.35)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Target Distribution by Wet / Dry Condition')
plt.xlabel('Target Residual')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'eda_wet_dry_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

display(train_eda.groupby('weather_group')['target'].agg(['count', 'mean', 'std']))
display(test_eda['weather_group'].value_counts().to_frame('count'))